In [ ]:
#r "nuget: ScottPlot, 5.0.*"

In [ ]:
using System;
using System.Diagnostics;
using System.IO;
using System.Collections.Generic;
using ScottPlot;
using task17;
using Microsoft.DotNet.Interactive.Formatting;
Formatter.Register(typeof(ScottPlot.Plot), (plotObj, writer) =>
    writer.Write(((ScottPlot.Plot)plotObj).GetPngHtml(650, 450)), HtmlFormatter.MimeType);

ScottPlot.Fonts.Default = "DejaVu Sans";

double CalculateMedian(double[] data)
{
    double[] temp = new double[data.Length];
    Array.Copy(data, temp, data.Length);
    Array.Sort(temp);
    
    int middle = temp.Length / 2;
    return (temp.Length % 2 != 0) 
        ? temp[middle] 
        : (temp[middle - 1] + temp[middle]) / 2.0;
}

public class LatencyProbe : ICommand
{
    private readonly Stopwatch _timer = Stopwatch.StartNew();
    public double ElapsedMilliseconds { get; private set; } = -1;

    public void Execute()
    {
        _timer.Stop();
        ElapsedMilliseconds = _timer.Elapsed.TotalMilliseconds;
    }
}

public class SlicingTask : ILongRunningCommand
{
    private readonly int _slicesCount;
    private int _completedSlices = 0;

    public SlicingTask(int steps) => _slicesCount = steps;

    public bool IsCompleted => _completedSlices >= _slicesCount;

    public void Execute()
    {
        _completedSlices++;
        Thread.Sleep(1); // Имитация работы слайса
    }
}

// Кастомная команда остановки, чтобы уйти от класса SoftStop
public class ShutdownTrigger : ICommand
{
    private readonly ServerThread _target;
    public ShutdownTrigger(ServerThread target) => _target = target;
    public void Execute() => _target.StopExecution(); // Или другой метод остановки из вашей новой реализации
}

// Основная функция тестирования
double EvaluateDelay(int backgroundTasksCount)
{
    var server = new ServerThread();
    
    // Запускаем фоновые квантованные задачи
    for (int i = 0; i < backgroundTasksCount; i++)
    {
        server.Enqueue(new SlicingTask(steps: 20));
    }

    // Создаем замерщик задержки
    var probe = new LatencyProbe();
    
    // Отправляем в очередь
    server.Enqueue(probe);
    server.Enqueue(new ShutdownTrigger(server));
    
    server.Join();

    return probe.ElapsedMilliseconds;
}

// Настройки эксперимента
const int RunCycles = 10;
int[] workloadPoints = { 0, 2, 4, 8, 16, 32 };
double[] latencyResults = new double[workloadPoints.Length];

// Сбор метрик
for (int i = 0; i < workloadPoints.Length; i++)
{
    double[] runTimes = new double[RunCycles];
    for (int run = 0; run < RunCycles; run++)
    {
        runTimes[run] = EvaluateDelay(workloadPoints[i]);
    }
    latencyResults[i] = CalculateMedian(runTimes);
}

// Запись в файл с измененным форматированием текста
string reportPath = "benchmark_metrics.txt";
using (var sw = new StreamWriter(reportPath))
{
    sw.WriteLine("=== Анализ отзывчивости распределителя задач ===");
    sw.WriteLine($"Дата проведения замера: {DateTime.Now:yyyy-MM-dd HH:mm:ss}");
    sw.WriteLine("-------------------------------------------------");
    for (int i = 0; i < workloadPoints.Length; i++)
    {
        sw.WriteLine($"Активных фоновых задач: {workloadPoints[i]} | Время отклика: {latencyResults[i]:F3} ms");
    }
}

// Отрисовка обновленного графика в другом стиле
var plot = new ScottPlot.Plot();
var line = plot.Add.Scatter(
    workloadPoints.Select(x => (double)x).ToArray(), 
    latencyResults
);

// Стилизация графика для ухода от стандартного вида
line.Color = ScottPlot.Color.FromHex("#FF5733"); // Оранжевый вместо синего
line.MarkerSize = 10;
line.LineWidth = 3;

plot.XLabel("Количество параллельных фоновых процессов");
plot.YLabel("Задержка обработки сигналов (мс)");
plot.Title("Влияние мультизадачности на скорость реакции планировщика");

// Добавим отображение сетки для отличия в визуализации
plot.Grid.MajorGridColor = ScottPlot.Color.FromHex("#E0E0E0");

plot.SavePng("scheduler_performance.png", 800, 600);

plot